# Step 1 — Train Per-Channel Noise Models

This notebook trains one Gaussian Mixture noise model per channel using:
1. **Noise2Void (N2V)** — self-supervised denoising to obtain a clean reference
2. **GaussianMixtureNoiseModel** — fit to (noisy, denoised) pixel pairs

**Run `00_datasets.ipynb` first** to create the training dataset.

For full-scale HPC runs, see `../cpg0000-jump-pilot/noise_models.sh`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '../../../src')  # point at JUMP-MicroSplit/src

DATASET_DIR    = Path('./cpg0000_training_dataset')
NOISE_MODELS_DIR = DATASET_DIR / 'noise_models'
CHANNELS       = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f'{DATASET_DIR} not found. Run 00_datasets.ipynb first.'
    )

NOISE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset:     {DATASET_DIR}')
print(f'Noise models: {NOISE_MODELS_DIR}')

## Train noise models (one per channel)

The `train_noise_model_for_channel` function runs Noise2Void + GMM fitting.
For a small notebook dataset we use reduced N2V epochs.

In [ ]:
from microsplit_reproducibility.workflows.noise_model import train_noise_model_for_channel

for channel in CHANNELS:
    nm_path = NOISE_MODELS_DIR / f'noise_model_{channel}.npz'
    if nm_path.exists():
        print(f'[{channel}] Already trained, skipping.')
        continue

    print(f'\n{'='*60}')
    print(f'Training noise model for channel: {channel}')
    print(f'{'='*60}')

    train_noise_model_for_channel(
        channel=channel,
        dataset_dir=str(DATASET_DIR),
        output_dir=str(NOISE_MODELS_DIR),
        n2v_epochs=5,   # fewer epochs for the notebook demo
        nm_epochs=50,
    )

print('\nAll noise models trained!')

## Inspect trained noise models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CHANNELS), figsize=(18, 4))

for ax, channel in zip(axes, CHANNELS):
    nm_path = NOISE_MODELS_DIR / f'noise_model_{channel}.npz'
    if not nm_path.exists():
        ax.set_title(f'{channel}\n(missing)')
        ax.axis('off')
        continue

    nm = np.load(str(nm_path), allow_pickle=True)
    print(f'{channel}: keys = {list(nm.keys())}')

    # Show noise model parameters summary
    ax.set_title(f'{channel}\n{nm_path.stat().st_size // 1024} KB')
    ax.axis('off')
    ax.text(0.5, 0.5, '✓ Trained', ha='center', va='center',
            fontsize=14, color='green', transform=ax.transAxes)

plt.suptitle('Noise model status (one .npz per channel)')
plt.tight_layout()
plt.show()

## Next step

Open **02_train.ipynb** to train the MicroSplit model.